# IaC & CI/CD

> **适用场景**: 基础设施自动化、数据管道部署
> **面试频率**: ⭐⭐⭐⭐ 高频

## 目录
1. Terraform: state / plan / apply
2. GitHub Actions / GitLab CI 基础
3. 环境隔离策略 dev/staging/prod
4. 练习题

---
## 1. Terraform: state / plan / apply

### 核心概念

**State（状态文件）**
- Terraform 用 `terraform.tfstate` 记录**当前管理的基础设施状态**
- 对比 state 与实际资源，确定需要的变更
- **生产环境必须用远端存储**（S3/GCS/Terraform Cloud），不能用本地文件

```hcl
# 远端 state 配置（S3）
terraform {
  backend "s3" {
    bucket         = "company-terraform-state"
    key            = "data-platform/prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-lock"  # 防止并发操作
  }
}
```

### 工作流程
```bash
terraform init      # 初始化，下载 provider 插件
terraform plan      # 预览变更（不执行），输出 diff
terraform apply     # 执行变更（需确认）
terraform destroy   # 销毁所有管理的资源

# 常用选项
terraform plan -out=plan.tfplan    # 保存 plan
terraform apply plan.tfplan        # 执行保存的 plan（不再需要确认）
terraform apply -auto-approve      # CI/CD 中跳过确认
terraform plan -target=module.airflow  # 只计划特定资源
```

### 数据工程常用资源示例
```hcl
# 创建 GCS bucket（数据湖）
resource "google_storage_bucket" "data_lake" {
  name          = "${var.project}-data-lake-${var.env}"
  location      = "US"
  storage_class = "STANDARD"

  lifecycle_rule {
    condition { age = 90 }
    action { type = "SetStorageClass"; storage_class = "NEARLINE" }
  }
}

# BigQuery dataset
resource "google_bigquery_dataset" "analytics" {
  dataset_id = "analytics_${var.env}"
  location   = "US"
  default_table_expiration_ms = null
}
```

### State 常见操作
```bash
terraform state list                    # 列出所有管理的资源
terraform state show google_storage_bucket.data_lake  # 查看资源详情
terraform state rm resource.name       # 从 state 移除（不删除实际资源）
terraform import resource.name <id>    # 导入已存在的资源到 state
```

---
## 2. GitHub Actions / GitLab CI 基础

### GitHub Actions 结构
```yaml
# .github/workflows/dbt_ci.yml
name: dbt CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  dbt-test:
    runs-on: ubuntu-latest
    environment: staging

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install dbt-bigquery==1.7.0

      - name: dbt compile (lint)
        run: dbt compile --profiles-dir . --target staging
        env:
          GOOGLE_APPLICATION_CREDENTIALS: ${{ secrets.GCP_SA_KEY }}

      - name: dbt test
        run: dbt test --profiles-dir . --target staging
        env:
          GOOGLE_APPLICATION_CREDENTIALS: ${{ secrets.GCP_SA_KEY }}

  deploy-prod:
    needs: dbt-test            # 依赖测试通过
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    environment: production    # 需要审批

    steps:
      - uses: actions/checkout@v4
      - name: dbt run
        run: dbt run --profiles-dir . --target prod
        env:
          GOOGLE_APPLICATION_CREDENTIALS: ${{ secrets.GCP_SA_KEY_PROD }}
```

### GitLab CI 结构
```yaml
# .gitlab-ci.yml
stages:
  - test
  - deploy

variables:
  DBT_PROFILES_DIR: .

dbt-test:
  stage: test
  image: python:3.11
  script:
    - pip install dbt-snowflake
    - dbt compile --target staging
    - dbt test --target staging
  only:
    - merge_requests
    - main

deploy-prod:
  stage: deploy
  script:
    - dbt run --target prod
  only:
    - main
  when: manual   # 需要手动触发
  environment:
    name: production
```

### Secrets 管理
- GitHub：Settings → Secrets and variables → Actions
- 使用 `${{ secrets.SECRET_NAME }}` 引用
- 生产环境用 Environment secrets（需要审批人批准才能使用）

---
## 3. 环境隔离策略 dev/staging/prod

### 三环境职责

| 环境 | 职责 | 数据 | 变更触发 |
|------|------|------|----------|
| **dev** | 开发测试，快速迭代 | 采样数据或模拟数据 | 手动或开发分支 push |
| **staging** | 集成测试，验证生产前 | 生产数据快照（脱敏） | PR 合并到 main |
| **prod** | 线上服务 | 真实生产数据 | 审批后手动触发 |

### dbt 多环境配置
```yaml
# profiles.yml
my_project:
  target: dev   # 默认目标
  outputs:
    dev:
      type: bigquery
      project: my-project-dev
      dataset: dbt_{{ env_var('DBT_USER', 'local') }}  # 每人独立 schema
      threads: 4

    staging:
      type: bigquery
      project: my-project-staging
      dataset: analytics_staging
      threads: 8

    prod:
      type: bigquery
      project: my-project-prod
      dataset: analytics
      threads: 16
```

### Terraform Workspace 实现环境隔离
```bash
# 创建和切换 workspace
terraform workspace new staging
terraform workspace select prod
terraform workspace list

# 在 HCL 中使用 workspace 变量
locals {
  env = terraform.workspace  # "dev" / "staging" / "prod"
  is_prod = terraform.workspace == "prod"
}

resource "google_bigquery_dataset" "main" {
  dataset_id = "analytics_${local.env}"
}
```

### 环境隔离最佳实践
1. **独立账号/项目**：prod 使用独立的云项目（GCP project / AWS account），而非仅靠命名区分
2. **最小权限原则**：CI/CD 的 Service Account 只有该环境的写权限
3. **数据脱敏**：staging 使用脱敏数据（不含真实 PII）
4. **部署流程**：dev → staging（自动）→ prod（人工审批）
5. **Rollback 能力**：保留上一个版本的部署，快速回滚

---
## 4. 练习题

### Q1 [高频] Terraform state 为什么要存在远端？本地存储有什么问题？

<details><summary>参考答案</summary>

**本地 state 的问题**：
1. **团队协作**：多人同时运行 terraform 可能导致 state 冲突
2. **丢失风险**：本地文件丢失则 Terraform 不知道哪些资源已经存在，会尝试重复创建
3. **无锁机制**：两个人同时 apply 会互相覆盖 state

**远端 state 的优势**：
- DynamoDB（AWS）或 GCS 对象锁 提供并发锁，防止同时操作
- 历史版本（S3 versioning）支持回滚
- 团队共享状态，CI/CD 可以统一访问
</details>

---

### Q2 CI/CD 中如何安全管理数据库密码和 API key？

<details><summary>参考答案</summary>

**不要做的**：
- 硬编码在代码/配置文件中
- 明文存在环境变量文件（.env 提交到 git）

**正确做法**：
1. **CI/CD Secrets**：GitHub Actions Secrets / GitLab CI Variables（Masked）
2. **Secret Manager**：AWS Secrets Manager / GCP Secret Manager，在运行时动态获取
3. **OIDC（无密钥认证）**：GitHub Actions OIDC + AWS IAM Role，无需存储长期密钥
4. **Service Account**：每个环境专用 SA，最小权限
5. **Vault**：HashiCorp Vault 统一 secrets 管理
</details>

---

### Q3 [高频] dev/staging/prod 三个环境的数据如何隔离？

<details><summary>参考答案</summary>

**推荐方案（从强到弱）**：
1. **独立云账号/项目**：完全隔离网络、权限、计费（最安全）
2. **独立数据库/Dataset/Schema**：同一云账号但不同数据库，通过 IAM 控制访问
3. **命名前缀区分**：`analytics_dev`, `analytics_staging`, `analytics_prod`（最弱）

**数据内容**：
- dev：生成的假数据或极小采样（< 1%）
- staging：生产数据快照，PII 字段脱敏（邮箱、手机号等替换）
- prod：完整真实数据，最严格权限控制
</details>

---

### Q4 如何设计一个 dbt 项目的 CI/CD 流程？

<details><summary>参考答案</summary>

**PR 阶段**（自动）：
1. `dbt compile` — 检查 SQL 语法
2. `dbt test --select state:modified+` — 只测试改动的模型及下游
3. SQLFluff lint（可选）
4. 在 PR comment 显示测试结果

**合并到 main**（自动部署到 staging）：
1. `dbt run --target staging`
2. `dbt test --target staging`
3. `dbt source freshness`

**部署到 prod**（人工审批）：
1. 审批人 review staging 结果
2. `dbt run --target prod --select state:modified+`（增量部署）
3. 运行 smoke tests
4. 失败则自动回滚
</details>